# Component 4 – Tea Maturity Classifier Training

**Function 1:** UAV RGB image → maturity class (Young / Medium / Mature) + confidence.

Runtime → Change runtime type → **T4 GPU** before running.

Expected in Google Drive: `MyDrive/C4_data/training_pack.zip`, made locally with
`python -m src.preprocess.make_training_pack` (contains `labels.csv` + `images/`).

## 1. Mount Drive and get the code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# If the repo is private, create a GitHub token (Settings → Developer settings → Fine-grained token,
# read-only, this repo only) and paste it when asked. Never save the token in the notebook.
import getpass, subprocess
REPO = 'https://github.com/ShiharaXcor/AI-Driven-Drone-System-for-Precision-Spraying-in-Sri-Lankan-Tea-Plantations-J26-DS-346.git'
BRANCH = 'component4/maturity-spray-engine'
token = getpass.getpass('GitHub token (leave empty if repo is public): ')
url = REPO.replace('https://', f'https://{token}@') if token else REPO
subprocess.run(['git', 'clone', '-q', '--branch', BRANCH, url, '/content/repo'], check=True)
del token, url
%cd /content/repo/component4_maturity_engine
!git log --oneline -3

In [ ]:
!pip -q install timm scikit-learn

## 2. Copy data from Drive

In [ ]:
DATA = '/content/drive/MyDrive/C4_data'
!rm -rf data/pack && mkdir -p data/pack
!unzip -q -o $DATA/training_pack.zip -d data/pack
!ls data/pack/images | wc -l && head -3 data/pack/labels.csv

## 3. Check the split (by source photo, stratified)

In [ ]:
from pathlib import Path
from src.classifier.dataset import load_labelled, group_split, describe
items = load_labelled(Path('data/pack/labels.csv'), Path('data/pack/images'))
print(describe(group_split(items)))

## 4. Train EfficientNet-B0 (main model)

In [ ]:
!python -m src.classifier.train --labels data/pack/labels.csv --tiles data/pack/images \
    --model efficientnet_b0 --epochs 25 --batch 32 --out runs/effnet_b0

## 5. Train MobileNetV3 (comparison model)

In [ ]:
!python -m src.classifier.train --labels data/pack/labels.csv --tiles data/pack/images \
    --model mobilenetv3_large_100 --epochs 25 --batch 32 --out runs/mobilenetv3

## 6. Compare models and view results

In [ ]:
import json, pandas as pd
from IPython.display import Image, display
rows = []
for run in ['runs/effnet_b0', 'runs/mobilenetv3']:
    r = json.load(open(f'{run}/results.json'))
    rows.append({'model': r['model'], 'accuracy': r['accuracy'], 'macro_precision': r['macro_precision'],
                 'macro_recall': r['macro_recall'], 'macro_f1': r['macro_f1'],
                 'young_recall': r['per_class']['young']['recall'],
                 'latency_ms_per_tile': r['latency_ms_per_tile']})
    display(Image(f'{run}/confusion_matrix.png'))
    display(Image(f'{run}/training_curves.png'))
pd.DataFrame(rows)

## 7. Save results back to Drive

In [ ]:
!mkdir -p $DATA/runs && cp -r runs/* $DATA/runs/
!ls -R $DATA/runs | head

## 8. Function 2 – spray prescription vs fixed-rate baseline

Pull the latest code, predict every **test** image with the best model, then compare the adaptive engine with conventional fixed-rate spraying.

If you restarted the runtime, run sections 1–2 first. No retraining needed: the model is loaded from Drive.

In [ ]:
!git pull -q
import os
if not os.path.exists('runs/effnet_b0/best.pt'):
    !mkdir -p runs && cp -r $DATA/runs/* runs/
!pip -q install pyyaml
!python -m src.classifier.predict --model runs/effnet_b0/best.pt \
    --split runs/effnet_b0/splits.csv --images data/pack/images \
    --out runs/effnet_b0/test_predictions.csv
!python -m src.prescription.evaluate --predictions runs/effnet_b0/test_predictions.csv \
    --out runs/effnet_b0/baseline_comparison

In [ ]:
import json, pandas as pd
from IPython.display import Image, display
r = json.load(open('runs/effnet_b0/baseline_comparison/comparison.json'))
display(Image('runs/effnet_b0/baseline_comparison/comparison.png'))
pd.DataFrame({k: r[k] for k in ['fixed_rate_baseline', 'adaptive_engine']}).T

In [ ]:
# sample prescription records (what goes to the sprayer / Component 2)
pd.read_csv('runs/effnet_b0/baseline_comparison/test_prescription.csv').head(10)

In [ ]:
!cp -r runs/effnet_b0/test_predictions.csv runs/effnet_b0/baseline_comparison $DATA/runs/effnet_b0/
!ls $DATA/runs/effnet_b0